In [1]:
import pandas as pd
import numpy as np
import os

os.listdir(".")

['brfss_env',
 'LLCP2024.XPT',
 'LLCP2021.XPT',
 'LLCP2020.XPT',
 'LLCP2022.XPT',
 'LLCP2023.XPT',
 'brfss_preprocess.ipynb']

In [2]:
files = {
    2020: "LLCP2020.XPT",
    2021: "LLCP2021.XPT",
    2022: "LLCP2022.XPT",
    2023: "LLCP2023.XPT",
    2024: "LLCP2024.XPT"
}

for year, f in files.items():
    df = pd.read_sas(f, format="xport", encoding="latin1")
    print(f"{year}: {df.shape}")

2020: (401958, 279)
2021: (438693, 303)
2022: (445132, 328)
2023: (433323, 350)
2024: (457670, 301)


In [7]:
def preprocess_brfss(file, year):

    df = pd.read_sas(file, format="xport", encoding="latin1")

    # Fix income variable name
    # 2020 used _INCOMG (5 bins, max 50k+) — not comparable to 2021+ _INCOMG1 (7 bins)
    # Decision: remove missing/refused (code 9), then set to NaN
    # See analysis2.ipynb for full cost/benefit justification
    if year == 2020:
        df = df.rename(columns={"_INCOMG": "_INCOMG1"})
        df = df[df["_INCOMG1"] < 9]  # remove missing/refused before setting NaN
        df["_INCOMG1"] = np.nan

    # Fix race variable
    # We use _RACEPRV (CDC recommended prevalence variable) over _RACE because:
    # 1. Derived from _RACE + _IMPRACE — fills missing values via imputation
    # 2. CDC renamed _RACE to _RACE1 in 2022 — _RACEPRV/_RACEPR1 follows consistent pattern
    # 3. CDC recommended variable for internet prevalence tables
    # Harmonized to 7 categories: 1=NH-White, 2=NH-Black, 3=AIAN, 4=Asian,
    # 5=NHOPI, 6=Other/Multiracial, 7=Hispanic
    # NOTE: In 2022 Other and Multiracial are combined into code 6 by CDC
    # We apply same collapse to all years for consistency
    if year == 2022:
        df = df.rename(columns={"_RACEPR1": "_RACEPRV"})
        # 2022 _RACEPR1 already uses 7-category scheme, no recoding needed
    else:
        # Recode _RACEPRV to match 2022 scheme:
        # Multiracial (7) -> Other/Multiracial (6)
        # Hispanic (8) -> 7
        df["_RACEPRV"] = df["_RACEPRV"].replace({7: 6, 8: 7})

    cols_needed = [
        "_STATE",
        "_LLCPWT",
        "_STSTR",   # stratification variable for complex sampling
        "_PSU",     # primary sampling unit for complex sampling
        "_BMI5",
        "_AGEG5YR",
        "_SEX",
        "_EDUCAG",
        "_INCOMG1",
        "_RACEPRV"
    ]

    df = df[cols_needed]

    # Remove invalid BMI first before any calculation
    # 9000+ = missing/refused, <=1200 = implausible (BMI <= 12)
    df = df[(df["_BMI5"] < 9000) & (df["_BMI5"] > 1200)]

    # Convert BMI (stored as integer x 100 in raw data)
    df["BMI"] = df["_BMI5"] / 100

    # Obesity indicator
    df["obese"] = (df["BMI"] >= 30).astype(int)

    # Remove missing/refused demographics
    # _INCOMG1: valid codes 1-7, code 9 = missing
    # _RACEPRV: valid codes 1-7, code 9 = missing
    # 2020 income is NaN by design — excluded from income filter
    df = df[
        (df["_AGEG5YR"] < 14) &
        (df["_SEX"] < 3) &
        (df["_EDUCAG"] < 5) &
        (df["_RACEPRV"] < 8) &
        ((df["_INCOMG1"] < 8) | (year == 2020))
    ]

    df["year"] = year

    return df

In [8]:
brfss_2020 = preprocess_brfss("LLCP2020.XPT", 2020)
brfss_2021 = preprocess_brfss("LLCP2021.XPT", 2021)
brfss_2022 = preprocess_brfss("LLCP2022.XPT", 2022)
brfss_2023 = preprocess_brfss("LLCP2023.XPT", 2023)
brfss_2024 = preprocess_brfss("LLCP2024.XPT", 2024)

print(brfss_2020.shape)
print(brfss_2021.shape)
print(brfss_2022.shape)
print(brfss_2023.shape)
print(brfss_2024.shape)

(300259, 13)
(320968, 13)
(326682, 13)
(326419, 13)
(348171, 13)


In [5]:
# Check 2020 income values
print("2020 income value counts:")
print(brfss_2020["_INCOMG1"].value_counts(dropna=False))

2020 income value counts:
_INCOMG1
NaN    355803
Name: count, dtype: int64


In [6]:
# Load raw 2020 to see original income distribution
df_2020_raw = pd.read_sas("LLCP2020.XPT", format="xport", encoding="latin1")
print("Original 2020 _INCOMG distribution:")
print(df_2020_raw["_INCOMG"].value_counts().sort_index())

Original 2020 _INCOMG distribution:
_INCOMG
1.0     26608
2.0     48767
3.0     31410
4.0     43851
5.0    171265
9.0     80057
Name: count, dtype: int64


In [9]:
brfss_all = pd.concat([
    brfss_2020,
    brfss_2021,
    brfss_2022,
    brfss_2023,
    brfss_2024
], ignore_index=True)

print("Total shape:", brfss_all.shape)

# Adjust survey weights for multi-year pooling
# Per CDC documentation, weights must be scaled proportionally
# when combining multiple years of BRFSS data
year_counts = brfss_all["year"].value_counts().sort_index()
total_n = len(brfss_all)

print("\nYear counts:")
print(year_counts)

year_props = year_counts / total_n
print("\nYear proportions:")
print(year_props.round(4))

brfss_all["_LLCPWT_adjusted"] = brfss_all.apply(
    lambda row: row["_LLCPWT"] * year_props[row["year"]], axis=1
)

print("\nOriginal weight stats:")
print(brfss_all["_LLCPWT"].describe().round(2))
print("\nAdjusted weight stats:")
print(brfss_all["_LLCPWT_adjusted"].describe().round(2))

# Key checks
print("\n_RACEPRV value counts by year:")
print(brfss_all.groupby("year")["_RACEPRV"].value_counts().unstack().fillna(0).astype(int))

print("\n_INCOMG1 value counts:")
print(brfss_all["_INCOMG1"].value_counts().sort_index())

Total shape: (1622499, 13)

Year counts:
year
2020    300259
2021    320968
2022    326682
2023    326419
2024    348171
Name: count, dtype: int64

Year proportions:
year
2020    0.1851
2021    0.1978
2022    0.2013
2023    0.2012
2024    0.2146
Name: count, dtype: float64

Original weight stats:
count    1622499.00
mean         576.39
std         1178.22
min            0.01
25%          104.98
50%          259.55
75%          596.87
max        72989.09
Name: _LLCPWT, dtype: float64

Adjusted weight stats:
count    1622499.00
mean         115.32
std          234.65
min            0.00
25%           21.01
50%           52.02
75%          119.62
max        14039.71
Name: _LLCPWT_adjusted, dtype: float64

_RACEPRV value counts by year:
_RACEPRV     1.0    2.0   3.0   4.0   5.0    6.0    7.0
year                                                   
2020      229065  22338  5244  7398  1611   8831  25772
2021      245069  23704  5533  8132  1497   9868  27165
2022      247265  25654  5294  93

In [10]:
brfss_all.to_csv("brfss_clean_2020_2024.csv", index=False)
print("saved.")
print("Columns:", brfss_all.columns.tolist())

saved.
Columns: ['_STATE', '_LLCPWT', '_STSTR', '_PSU', '_BMI5', '_AGEG5YR', '_SEX', '_EDUCAG', '_INCOMG1', '_RACEPRV', 'BMI', 'obese', 'year', '_LLCPWT_adjusted']


In [11]:
race_map = {
    1: "NH-White", 2: "NH-Black", 3: "AIAN",
    4: "Asian", 5: "NHOPI", 6: "Other/Multiracial",
    7: "Hispanic"
}
age_map = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39",
    5: "40-44", 6: "45-49", 7: "50-54", 8: "55-59",
    9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"
}
sex_map = {1: "Male", 2: "Female"}
education_map = {
    1: "Did not graduate high school",
    2: "Graduated high school",
    3: "Attended college or technical school",
    4: "Graduated college or technical school"
}
income_map = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}

brfss_all["age_group"]    = brfss_all["_AGEG5YR"].map(age_map)
brfss_all["sex"]          = brfss_all["_SEX"].map(sex_map)
brfss_all["education"]    = brfss_all["_EDUCAG"].map(education_map)
brfss_all["income_group"] = brfss_all["_INCOMG1"].map(income_map)
brfss_all["race_group"]   = brfss_all["_RACEPRV"].map(race_map)

print("Race groups:", sorted(brfss_all["race_group"].dropna().unique()))
print("Income groups:", sorted(brfss_all["income_group"].dropna().unique()))
print("Age groups:", sorted(brfss_all["age_group"].dropna().unique()))

Race groups: ['AIAN', 'Asian', 'Hispanic', 'NH-Black', 'NH-White', 'NHOPI', 'Other/Multiracial']
Income groups: ['100k-200k', '15k-25k', '200k+', '25k-35k', '35k-50k', '50k-100k', '<15k']
Age groups: ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']


In [12]:
group_cols = ["age_group", "sex", "education", "income_group", "race_group"]

def weighted_obesity_rate(group):
    return np.average(group["obese"], weights=group["_LLCPWT_adjusted"])

brfss_group_summary = (
    brfss_all.groupby(group_cols)
      .apply(weighted_obesity_rate)
      .reset_index()
)
brfss_group_summary.columns = group_cols + ["obesity_rate"]

# Add unweighted n for sample size reference
n_counts = brfss_all.groupby(group_cols).size().reset_index(name="n")
brfss_group_summary = brfss_group_summary.merge(n_counts, on=group_cols)

# Reliable flag
brfss_group_summary["reliable"] = (brfss_group_summary["n"] >= 30).astype(int)

# Smoothed rate
global_mean = brfss_group_summary["obesity_rate"].mean()
brfss_group_summary["obesity_rate_smoothed"] = (
    (brfss_group_summary["n"] * brfss_group_summary["obesity_rate"] + 30 * global_mean) /
    (brfss_group_summary["n"] + 30)
)

brfss_group_summary = brfss_group_summary.sort_values("n", ascending=False)

print(brfss_group_summary.shape)
print(f"Reliable cells: {brfss_group_summary['reliable'].sum()}")
print(f"Sparse cells: {(brfss_group_summary['reliable']==0).sum()}")

brfss_group_summary.to_csv("brfss_group_summary.csv", index=False)
print("saved.")

(4953, 9)
Reliable cells: 2893
Sparse cells: 2060
saved.
